In [7]:
import os
import sys
import json
import torch
from pathlib import Path
from dotenv import load_dotenv

# 1. Load security tokens
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

# 2. Path Detection (Adjust 'A-RICD' to your actual folder name if different)
BASE_DIR = Path(r"D:\Md. Al Baki Akon\A-RICD")
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# 3. Define Constants for the entire notebook
MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"
ADAPTER_PATH = BASE_DIR / "models" / "llama2-bio-checkpoints" / "checkpoint-636"
DATA_PATH = BASE_DIR / "data" / "evaluation_dataset" / "factscore" / "GPT-4.jsonl"
SAVE_PATH = BASE_DIR / "exp_results" / "factscore" / "A_RICD_Llama2_7b_results.jsonl"

print(f"✓ Root Directory: {BASE_DIR}")
print(f"✓ Target Adapter: {ADAPTER_PATH}")
print(f"✓ CUDA Device: {torch.cuda.get_device_name(0)}")

✓ Root Directory: D:\Md. Al Baki Akon\A-RICD
✓ Target Adapter: D:\Md. Al Baki Akon\A-RICD\models\llama2-bio-checkpoints\checkpoint-636
✓ CUDA Device: NVIDIA GeForce RTX 4090


In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Initialize MCP Client
mcp_client = MultiServerMCPClient({
    "wikipedia": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "wikipedia_mcp"]
    }
})

async def get_adaptive_context(topic):
    """Fetches Wikipedia grounding."""
    try:
        results = await mcp_client.call_tool("wikipedia", "search", {"keyword": topic})
        if results and "content" in results:
            return results["content"][:1800] 
        return "No context found."
    except Exception as e:
        return f"MCP Error: {e}"

print("✓ Pillar 1 (MCP) Initialized.")

✓ Pillar 1 (MCP) Initialized.


In [10]:
import torch
import gc

def clear_gpu_memory():
    # 1. Clear Python references
    gc.collect()
    # 2. Clear Torch Cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        # 3. Check Free Memory
        free_mem = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"✓ GPU Memory Cleared. Free VRAM: {free_mem:.2f} GB")

clear_gpu_memory()

✓ GPU Memory Cleared. Free VRAM: 20.45 GB


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

# 1. 4-Bit Configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 2. Load the Expert (Base Model)
print("Loading Expert Model...")
expert_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0}, # Forces the model entirely onto GPU 0
    token=os.environ.get("HF_TOKEN")
)

# 3. Load the Amateur (Base + Your Adapter)
print(f"Loading Amateur Adapter from: {ADAPTER_PATH}")
# We load the base specifically for the adapter to avoid sharing weights in memory
base_for_adapter = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0}, # Forces this onto GPU 0 as well
    token=os.environ.get("HF_TOKEN")
)

amateur_model = PeftModel.from_pretrained(
    base_for_adapter,
    str(ADAPTER_PATH)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
expert_model.eval()
amateur_model.eval()

print(f"✓ Success! Total VRAM used: {(torch.cuda.memory_allocated(0)/1024**3):.2f} GB")

Loading Expert Model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.63s/it]


Loading Amateur Adapter from: D:\Md. Al Baki Akon\A-RICD\models\llama2-bio-checkpoints\checkpoint-636


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.36s/it]


✓ Success! Total VRAM used: 7.81 GB


In [13]:
import json
import torch
from tqdm import tqdm # Standard terminal bar, no widgets needed

async def run_aricd_production_loop():
    # 1. Load the 500 topics
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        dataset = [json.loads(line) for line in f]
    
    # 2. Progress Tracking
    completed_topics = []
    if SAVE_PATH.exists():
        with open(SAVE_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    completed_topics.append(json.loads(line)['topic'])
                except: continue
    
    print(f"Total: {len(dataset)} | Done: {len(completed_topics)}")
    print(f"Remaining: {len(dataset) - len(completed_topics)}")

    # 3. Production Loop
    # Using the standard tqdm for stability
    for entry in tqdm(dataset, desc="A-RICD Progress"):
        topic = entry.get("topic")
        if topic in completed_topics:
            continue
            
        try:
            # Step A: Grounding via MCP
            context = await get_adaptive_context(topic)
            
            # Step B: Generation
            prompt = f"[INST] Biography of {topic}. Use context: {context} [/INST]"
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            
            with torch.no_grad():
                outputs = expert_model.generate(
                    **inputs, 
                    max_new_tokens=450,
                    temperature=0.1,
                    do_sample=True
                )
                bio = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Step C: Safety Save
            result = {"topic": topic, "output": bio, "method": "A-RICD"}
            with open(SAVE_PATH, 'a', encoding='utf-8') as out_f:
                out_f.write(json.dumps(result) + "\n")
            
            # Step D: Periodic VRAM Clear
            torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"\nError processing {topic}: {e}")
            continue

# Start the full run
await run_aricd_production_loop()

Total: 500 | Done: 0
Remaining: 500


A-RICD Progress: 100%|██████████| 500/500 [1:34:51<00:00, 11.38s/it]


In [17]:
import subprocess
import sys

def install_spacy_model():
    print("Downloading spaCy English model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    print("✓ spaCy model installed successfully.")

install_spacy_model()

✓ spaCy model installed successfully.


In [27]:
import json
import os
from pathlib import Path

# 1. Define the cache directory
demo_dir = Path(".cache/factscore/demos")
demo_dir.mkdir(parents=True, exist_ok=True)

# 2. A full set of 8 demonstrations to satisfy the FActScore library index
biography_demons = {
    "John Smith (born May 5, 1970) is an American architect.": [
        "John Smith was born on May 5, 1970.",
        "John Smith is an architect.",
        "John Smith is American."
    ],
    "She studied at Harvard and later joined NASA in 2005.": [
        "She studied at Harvard.",
        "She joined NASA.",
        "She joined NASA in 2005."
    ],
    "Einstein was a German-born theoretical physicist who developed the theory of relativity.": [
        "Einstein was born in Germany.",
        "Einstein was a theoretical physicist.",
        "Einstein developed the theory of relativity."
    ],
    "Born in Tokyo, she moved to London at the age of ten.": [
        "She was born in Tokyo.",
        "She moved to London.",
        "She moved to London when she was ten years old."
    ],
    "He won the Nobel Prize in Literature in 1954 for his mastery of the art of narrative.": [
        "He won the Nobel Prize in Literature.",
        "He won the Nobel Prize in 1954.",
        "He won the prize for his mastery of the art of narrative."
    ],
    "The band was formed in London in 1962 and became world famous.": [
        "The band was formed in London.",
        "The band was formed in 1962.",
        "The band became world famous."
    ],
    "He served as the 44th president of the United States from 2009 to 2017.": [
        "He was the 44th president of the United States.",
        "He served as president from 2009 to 2017."
    ],
    "She is a retired professional tennis player who held the world No. 1 ranking.": [
        "She is a professional tennis player.",
        "She is retired.",
        "She held the world No. 1 ranking."
    ]
}

# 3. Save to both required locations
for filename in ["demons.json", "demons_complex.json"]:
    with open(demo_dir / filename, "w", encoding="utf-8") as f:
        json.dump(biography_demons, f, indent=4)

print("✓ Successfully created 8 robust demonstrations. Index error should now be resolved.")

✓ Successfully created 8 robust demonstrations. Index error should now be resolved.


In [28]:
import nltk

# Download the specific missing resource for the new NLTK version
nltk.download('punkt_tab')

# Re-verify the standard punkt for safety
nltk.download('punkt')

print("✓ NLTK resources updated successfully.")

✓ NLTK resources updated successfully.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pciuc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pciuc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [31]:
import os
from pathlib import Path

# 1. Define a local path for the key file
key_file_path = Path("openai_key.txt")

# 2. Get the key from your environment variable
api_key = os.environ.get("OPENAI_API_KEY")

if api_key:
    # 3. Write it to the file (FActScore expects the key on the first line)
    with open(key_file_path, "w", encoding="utf-8") as f:
        f.write(api_key)
    print(f"✓ OpenAI key file created at {key_file_path}")
else:
    print("Error: OPENAI_API_KEY environment variable is not set.")

✓ OpenAI key file created at openai_key.txt


In [33]:
import factscore.openai_lm
import factscore.atomic_facts

# Monkey patch: Redirect the library from the deprecated 'davinci' to the active 'turbo-instruct'
# We also update the generation function to ensure it uses the correct API parameters
def patched_call_gpt(prompt, model_name="gpt-3.5-turbo-instruct", max_len=512, temp=0, num_log_probs=0, echo=False, verbose=False):
    import openai
    # Note: Using the legacy-compatible Completion API which gpt-3.5-turbo-instruct supports
    response = openai.Completion.create(
        model=model_name,
        prompt=prompt,
        max_tokens=max_len,
        temperature=temp,
        logprobs=num_log_probs,
        echo=echo
    )
    return response

# Apply the patch to the library's namespace
factscore.openai_lm.call_GPT3 = patched_call_gpt
print("✓ Patched FActScore to use gpt-3.5-turbo-instruct (Replacement for Davinci).")

✓ Patched FActScore to use gpt-3.5-turbo-instruct (Replacement for Davinci).


In [34]:
import os
import json
import pandas as pd
from factscore.factscorer import FactScorer

def run_final_thesis_audit():
    # Use the key file path we created previously
    key_path = "openai_key.txt"
    
    if not os.path.exists(key_path):
        print("Error: openai_key.txt not found. Please recreate the key file.")
        return

    # Initialize Scorer
    # The patched call_GPT3 will now handle the model transition automatically
    fs = FactScorer(openai_key=key_path)
    
    # Load your 500 A-RICD biographies
    results = []
    with open(SAVE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except: continue
    
    df = pd.DataFrame(results)
    
    print(f"✓ Starting FActScore audit for {len(df)} biographies using Patched GPT Model...")
    
    # Execute the audit
    # Gamma=10 matches the ICD paper's penalty for brevity
    out = fs.get_score(df['topic'].tolist(), df['output'].tolist(), gamma=10)
    
    print("\n" + "="*60)
    print("             OFFICIAL THESIS RESULTS: A-RICD             ")
    print("="*60)
    print(f"Final FActScore:     {round(out['score'] * 100, 2)}%")
    print(f"Avg Facts/Bio:       {round(out['num_facts_per_response'], 1)}")
    print(f"Total Atomic Facts:  {int(out['num_facts'])}")
    print("="*60)

run_final_thesis_audit()

✓ Starting FActScore audit for 500 biographies using Patched GPT Model...


CRITICAL:root:Estimated OpenAI API cost for atomic fact generation ($0.020 per 1000 tokens): $41.79 for 1567210 words and 2089613 tokens


RateLimitError: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.